# 05 — Sécurité dans le cloud (survol)

**Objectif :** **vue d'ensemble** des piliers de la sécurité Azure — identité, RBAC, secrets, réseau, données. On reste **en survol** : chaque sujet mériterait son propre parcours, l'idée ici est de savoir **de quoi on parle** et **où chercher** quand on en aura besoin.

À la fin :
- vous saurez expliquer **defense in depth**, **least privilege**, **zero trust** ;
- vous saurez ce qu'est **Microsoft Entra ID**, un **service principal**, une **managed identity** ;
- vous saurez à quoi sert **RBAC** et comment lire une **role assignment** ;
- vous saurez ce qu'est **Key Vault**, et comment on stocke un secret proprement ;
- vous aurez une vision de la sécurité **réseau** et **données** d'Azure ;
- vous saurez où vit **Defender for Cloud** + **Azure Policy**.

## 1. Les 3 principes à connaître par cœur

Toute la sécurité cloud s'articule autour de ces 3 idées.

### 🛡️ Defense in depth — *défense en profondeur*

On ne se repose **pas** sur une seule barrière. On empile : réseau + identité + chiffrement + monitoring + sauvegardes. Si une couche tombe, les autres tiennent.

```
  🌍 Internet
      │
      ▼  🛡️ couche 1 — Front Door + WAF (filtre les requêtes malveillantes)
      ▼  🛡️ couche 2 — réseau privé (VNet, Private Endpoint, NSG)
      ▼  🛡️ couche 3 — identité Entra ID + MFA
      ▼  🛡️ couche 4 — autorisation RBAC (least privilege)
      ▼  🛡️ couche 5 — chiffrement at rest & in transit
      ▼  🛡️ couche 6 — logs + alertes (Defender, App Insights)
      ▼  🛡️ couche 7 — sauvegardes & resilience
      📦 votre ressource
```

### 🎯 Least privilege — *moindre privilège*

Chaque identité a **le minimum de droits** pour faire son boulot. Si la Web App n'a besoin que de **lire** Key Vault, on lui donne `Key Vault Secrets User`, pas `Owner`.

### 🚫 Zero Trust — *ne fais confiance à personne*

Pas de notion de « réseau interne sûr ». Chaque requête est **vérifiée**, qu'elle vienne de l'extérieur ou de l'intérieur. Trois mantras :
1. **Verify explicitly** — toujours authentifier & autoriser.
2. **Use least privilege access**.
3. **Assume breach** — coder comme si l'attaquant était déjà à l'intérieur.

## 2. Identité — Microsoft Entra ID

**Microsoft Entra ID** (anciennement *Azure Active Directory*) est l'annuaire d'identités de Microsoft. C'est lui qui sait :
- qui sont les **utilisateurs** humains,
- les **groupes** auxquels ils appartiennent,
- les **applications** (= identités non-humaines) qui peuvent se connecter.

### Les types d'identités

| Identité | Pour qui ? | Exemple |
|----------|------------|---------|
| **User** | Un humain | Votre compte `leo@contoso.com` |
| **Group** | Un ensemble d'utilisateurs (ou d'autres groupes) | `devs-équipe-data` |
| **Service Principal** (SP) | Une application qui s'authentifie avec un **secret** ou un **certificat** | Un CI/CD qui pousse en prod |
| **Managed Identity** ⭐ | Une appli Azure qui s'authentifie **sans clé** auprès d'autres ressources Azure | Une Web App qui appelle Foundry |

**Managed Identity** est la voie royale dans Azure : **plus aucune clé à stocker**, Azure gère le cycle de vie du credential pour vous. À privilégier dès que c'est possible.

### Authentification multi-facteurs (MFA)

L'utilisateur prouve son identité **deux fois** : mot de passe + code SMS / app Authenticator / clé FIDO2. **Indispensable** pour tout compte admin.

### Conditional Access

Des règles du type *« si tel utilisateur, depuis tel pays, sur telle app → exiger MFA »* ou *« bloquer si pays à risque »*. Configurées par les admins du tenant.

## 3. Autorisation — RBAC

**RBAC** *(Role-Based Access Control)* répond à la question : *qui a le droit de faire quoi sur quoi ?*

### L'équation RBAC

```
  ┌──────────┐     ┌─────────┐     ┌────────────┐     ┌────────────────┐
  │ IDENTITÉ │  +  │  ROLE   │  +  │  SCOPE     │  =  │ Role Assignment│
  └──────────┘     └─────────┘     └────────────┘     └────────────────┘
     User /          Owner /         Subscription /
     Group /         Contrib. /      Resource Group /
     SP /            Reader /        Resource
     MI              custom
```

**Exemple concret** : *« Léo a le rôle Owner sur le RG `rg-stage-leo` »*  
→ identité = Léo, rôle = Owner, scope = RG `rg-stage-leo`.

### Les rôles built-in à connaître

| Rôle | Peut faire |
|------|-----------|
| **Owner** | Tout, **y compris donner des droits** à d'autres (gérer RBAC). |
| **Contributor** | Tout, **sauf** gérer RBAC. |
| **Reader** | Lire uniquement. |
| **User Access Administrator** | Gérer RBAC uniquement (donner des droits). |
| ... + des centaines de rôles spécifiques | Ex. `Storage Blob Data Reader`, `Key Vault Secrets User`, `Foundry User`, `Cognitive Services OpenAI User`. |

### L'héritage des scopes

Un droit donné à un scope **descend automatiquement** aux scopes enfants.

```
  Tenant
   └── Management Group         ← donner Reader ici …
        └── Subscription        ← … donne Reader ici …
             └── Resource Group ← … et ici …
                  └── Ressource ← … et ici.
```

**Conseil pratique** : donnez les droits au **scope le plus bas possible** (notebook 01 = un seul RG, pas l'abonnement entier).

### Vérifier ses droits — `az cli`

```bash
az role assignment list \
    --scope $(az group show --name rg-stage-leo --query id -o tsv) \
    --assignee $(az ad signed-in-user show --query id -o tsv) \
    -o table
```

C'est exactement ce qu'on a fait dans la section 5 du notebook 01.

## 4. 🎬 Mise en situation — *Web App → Foundry, sans clé*

**Objectif :** la Web App du notebook 01 doit appeler l'**agent Foundry** du notebook 02, **sans hardcoder de clé**. C'est LE cas d'usage de Managed Identity.

```
       ┌─────────────────────────┐                        ┌─────────────────────────┐
       │  Web App (App Service)  │                        │  Foundry (project)      │
       │                         │                        │                         │
       │  • code Python          │  ── 1. demande token ──►  Entra ID                │
       │  • identité managée     │  ◄── 2. token ─────────                           │
       │    (System-assigned)    │                        │                         │
       │                         │  ── 3. POST /chat ─────►│  endpoint agent         │
       │                         │       Authorization:    │  vérifie le token,      │
       │                         │       Bearer <token>    │  vérifie le rôle RBAC   │
       └─────────────────────────┘  ◄── 4. réponse ─────── │  → OK, exécute          │
                                                          │                         │
                                                          └─────────────────────────┘
```

### Les 3 étapes côté Azure

1. **Activer une Managed Identity sur la Web App :**
   ```bash
   az webapp identity assign --name $WEBAPP --resource-group $RG
   ```
   → Azure crée une identité dans Entra ID, dont le **lifecycle est lié à la Web App**.

2. **Donner à cette identité le rôle `Cognitive Services OpenAI User`** sur la ressource Foundry :
   ```bash
   MI_OBJECTID=$(az webapp identity show --name $WEBAPP --resource-group $RG --query principalId -o tsv)
   FOUNDRY_ID=$(az cognitiveservices account show --name $FOUNDRY --resource-group $RG --query id -o tsv)
   az role assignment create \
       --assignee $MI_OBJECTID \
       --role "Cognitive Services OpenAI User" \
       --scope $FOUNDRY_ID
   ```

3. **Côté code Python**, `DefaultAzureCredential` détecte automatiquement la Managed Identity et récupère un token :
   ```python
   from azure.identity import DefaultAzureCredential
   from azure.ai.projects import AIProjectClient

   project = AIProjectClient(
       endpoint=PROJECT_ENDPOINT,
       credential=DefaultAzureCredential(),   # ← magie : Entra ID + Managed Identity
   )
   # plus aucune clé dans le code, plus aucun .env
   ```

✅ **Aucune clé** dans le code source, dans Key Vault, ou ailleurs. La Web App peut appeler Foundry **uniquement** parce qu'elle est *elle-même*. Si vous supprimez la Web App, son identité disparaît.

> 🧠 `DefaultAzureCredential` essaie une chaîne de credentials :
> 1. Variables d'environnement (`AZURE_CLIENT_ID`, etc.) — utile pour CI/CD
> 2. Managed Identity (si dispo) — utile en prod sur Azure
> 3. `az login` — utile en dev local
> 4. VS Code / Azure CLI / etc.
> 
> Vous écrivez **le même code en dev et en prod**, il « branche » la bonne source automatiquement.

## 5. Secrets — Azure Key Vault

Quand on a **quand même** besoin d'un secret (clé d'API externe, chaîne de connexion à un service tiers…), on ne le met pas dans le code, pas dans un `.env`, pas dans une variable d'environnement de la Web App. On le met dans **Azure Key Vault**.

```
  ┌──────────────────────────────────────────────────────────────────┐
  │                       Azure Key Vault                            │
  │                                                                  │
  │  Secrets : strings sensibles (mots de passe, API keys, conn str) │
  │  Keys    : clés cryptographiques (RSA, EC) — chiffrement         │
  │  Certs   : certificats TLS                                       │
  └──────────────────────────────────────────────────────────────────┘
```

**Trois bonnes pratiques** :

1. **Lecture par Managed Identity**, jamais par clé d'accès.
2. **Audit complet** activé (logs de chaque lecture).
3. **Soft-delete + purge protection** activés (un secret supprimé peut être restauré).

### Exemple — l'App Service lit un secret au démarrage

Plutôt que coder en dur la chaîne de connexion SQL, on la stocke dans Key Vault et on **référence** depuis l'App Service :

```bash
# Setting de la Web App avec une référence Key Vault
!az webapp config appsettings set \
    --name $WEBAPP --resource-group $RG \
    --settings "SQL_CONN=@Microsoft.KeyVault(SecretUri=https://my-kv.vault.azure.net/secrets/sql-conn/)"
```

L'App Service récupère le secret à chaque redémarrage. Pas besoin de le voir passer.

> 🔄 **Rotation des secrets** : un secret qui ne change jamais finit par fuiter. Key Vault facilite la **rotation automatique** (notamment pour SQL et Storage).

## 6. Sécurité réseau (en 5 mots-clés)

Par défaut, beaucoup de PaaS Azure ont leur endpoint **public sur Internet**. Pour de la prod sensible, on les isole dans un réseau privé. Voici le vocabulaire :

| Concept | Idée |
|---------|------|
| **VNet** *Virtual Network* | Votre réseau privé dans Azure. |
| **Subnet** | Une sous-division d'un VNet. |
| **NSG** *Network Security Group* | Pare-feu attaché à un subnet : règles entrant/sortant. |
| **Private Endpoint** | Un PaaS (Foundry, SQL, Storage…) exposé **uniquement** via une IP privée du VNet. Internet ne le voit plus. |
| **WAF** *Web Application Firewall* | Filtre les requêtes web (injections SQL, XSS…), souvent monté avec Front Door ou App Gateway. |
| **DDoS Protection** | Anti-déni-de-service au niveau Azure. |

**Pour ce parcours**, on reste sur les endpoints publics (suffisant pour apprendre, et compatible avec votre RG existant). En production : Private Endpoint + WAF systématiquement.

## 7. Sécurité des données

Deux dimensions à toujours vérifier :

### 🔒 At rest — *au repos sur le disque*

Toutes les ressources Azure chiffrent les données stockées par défaut (**AES-256**). Sur les services-clés vous pouvez fournir **vos propres clés** (*Customer-Managed Keys / BYOK*) stockées dans Key Vault.

### 🚄 In transit — *pendant le transport*

**TLS 1.2 minimum** (idéalement 1.3) sur toutes les communications. HTTPS **obligatoire** sur App Service depuis 2024.

### 🤐 Données personnelles & sensibles

Pour les LLM : **ne jamais envoyer** de données ultra-sensibles (PII non-masquée, secrets, RIB…) dans un prompt sans réfléchir. Azure AI **Content Safety** + **PII detection** (Azure AI Language) aident à filtrer.

**RGPD** : si vous traitez des données européennes, choisissez une région européenne et activez les logs d'accès.

## 8. Gouvernance — qu'est-ce qu'on a le droit de faire ?

Au-delà du *qui peut faire quoi*, l'organisation veut imposer **quelles ressources** on peut créer.

### Azure Policy

Des règles déclaratives. Exemples :
- *« interdit de créer une VM en dehors de l'Europe »* ;
- *« obligatoire d'avoir un tag `cost-center` sur chaque ressource »* ;
- *« interdit les SKU Premium sur les abonnements dev »*.

Si vous essayez de créer une ressource non conforme, Azure refuse.

### Defender for Cloud

Le **CSPM** *(Cloud Security Posture Management)* de Microsoft. Il vous donne :
- un **Secure Score** entre 0 et 100 pour votre abonnement ;
- des **recommandations** priorisées (ex. *« activer MFA pour les comptes admin »*) ;
- des **alertes** en cas de comportement suspect (login depuis un pays inhabituel, exfiltration de données…).

### Microsoft Sentinel (SIEM)

Si vous voulez une plateforme SOC complète (logs, corrélation, hunting, réponse), c'est **Sentinel** (bâti sur Log Analytics).

### Locks

Empêcher la suppression accidentelle. Deux niveaux : `ReadOnly` et `CanNotDelete`.

```bash
az lock create --name no-delete-prod --resource-group rg-prod --lock-type CanNotDelete
```

## 9. Sécurité spécifique à l'IA — *Foundry*

Les agents IA ouvrent de nouvelles surfaces d'attaque. Foundry intègre :

| Risque | Parade Foundry |
|--------|----------------|
| **Prompt injection** : un utilisateur ou un document piège l'agent (*« ignore tes instructions et … »*) | **Content Safety** + **Prompt Shields** (filtres en entrée) |
| **Sorties dangereuses** (violence, haine, sexuel, automutilation) | **Content filters** sur les déploiements modèles (4 niveaux : low / medium / high / off) |
| **PII** dans les prompts ou réponses | **PII detection** (Azure AI Language) + masquage |
| **Hallucinations** | **Groundedness** evaluator (notebook 06) + RAG avec citations |
| **Jailbreaks** (contournements de policies) | **Jailbreak detection** dans Prompt Shields |
| **Fuite de données via tools** | RBAC strict sur les outils, **MCP server** scopé, OBO (On-Behalf-Of) pour propager l'identité utilisateur |

📚 Doc Responsible AI Foundry : https://learn.microsoft.com/azure/foundry/responsible-ai/

## 10. Checklist sécurité — *à demander à votre équipe*

Quand vous arrivez sur un projet cloud, posez ces 10 questions :

1. ✅ **Entra ID** activé pour tous les humains ? **MFA** obligatoire ?
2. ✅ **RBAC** au scope le plus fin possible (pas de `Owner` distribué à tout l'étage) ?
3. ✅ Les apps utilisent-elles des **Managed Identities** ou est-ce qu'on a encore des **clés en dur** quelque part ?
4. ✅ Les secrets (clés tierces, conn strings) sont-ils dans **Key Vault** avec rotation activée ?
5. ✅ Les services exposés sur Internet ont-ils un **WAF** devant + HTTPS obligatoire ?
6. ✅ La prod a-t-elle des **Private Endpoints** sur les BDD / Storage / Foundry ?
7. ✅ Les logs (Activity Log, App Insights) sont-ils **centralisés** et conservés assez longtemps (≥ 90 j) ?
8. ✅ **Defender for Cloud** activé ? Secure Score suivi mensuellement ?
9. ✅ Les Resource Groups critiques ont-ils un **lock** `CanNotDelete` ?
10. ✅ Y a-t-il un **runbook** pour les incidents (rotation de clé, suppression de compte, breach) ?

## Récap — Cheatsheet

| Terme | Vite dit |
|-------|----------|
| **Defense in depth** | On empile les barrières. |
| **Least privilege** | Le minimum de droits nécessaire. |
| **Zero Trust** | Vérifier explicitement à chaque requête. |
| **Entra ID** | L'annuaire d'identités Microsoft. |
| **User / Group / SP / MI** | Les 4 types d'identités. |
| **MFA** | Auth multi-facteurs. |
| **RBAC** | Identité + Rôle + Scope = Role Assignment. |
| **Owner / Contributor / Reader** | Les 3 rôles built-in généraux. |
| **Managed Identity** ⭐ | Identité gérée par Azure, pas de clé à gérer. |
| **Key Vault** | Coffre-fort : secrets, clés, certs. |
| **VNet / Subnet / NSG** | Réseau privé + pare-feu. |
| **Private Endpoint** | Un PaaS accessible uniquement en interne. |
| **WAF** | Pare-feu applicatif web. |
| **At rest / in transit** | Chiffrement disque / réseau. |
| **Azure Policy** | Règles déclaratives obligatoires. |
| **Defender for Cloud** | CSPM + Secure Score + alertes. |
| **Sentinel** | SIEM complet. |
| **Content Safety / Prompt Shields** | Garde-fous IA pour Foundry. |

**Et après ?**
- Notebook 06 — **Monitoring & évaluation** : Application Insights, KQL, eval Foundry.
- Pour pratiquer : mettre en place une Managed Identity Web App → Foundry (voir mise en situation §4).

📚 Pour aller plus loin :
- Microsoft *Cloud Adoption Framework* — sécurité : https://learn.microsoft.com/azure/cloud-adoption-framework/secure/
- *Zero Trust principles* : https://learn.microsoft.com/security/zero-trust/
- *Azure security baseline* (par service) : https://learn.microsoft.com/security/benchmark/azure/
- Certification **AZ-500** *Azure Security Engineer* : https://learn.microsoft.com/certifications/azure-security-engineer/
- *Responsible AI* Foundry : https://learn.microsoft.com/azure/foundry/responsible-ai/